# Known-input tree context — v005

CPU-only, A-only controlled feature comparison using approved Gym data. Default execution reviews saved records without importing LightGBM, reading assignment CSVs, fitting, or writing artifacts. Set `PA3_RUN_TREE_V005=1` to run and `PA3_RESUME_TREE_V005=1` to reuse compatible completed records. All modeling code is contained here.

The existing 20-feature / 511-leaf / 2,500-tree A model is verified and reused. Three fresh arms add offline known-input context (34 columns), causal dynamics proxies (33), or both (47). R/C remain numeric. Each fresh arm has exactly the same fixed 2,500-tree budget, parameters, seed, and original 61,114 fit / 6,791 validation breath partition. No validation feedback, early stopping, B/C scoring, final all-labeled fitting, or submission occurs.

Offline context explicitly assumes every test breath's known inputs are available. Pressure, row IDs, and breath IDs never become features. Valve opening is a control proxy, not measured flow or volume. No external competition data or solutions are used.


In [ ]:
import os, gc, csv, hashlib, json, time, uuid, shutil, fcntl
from pathlib import Path
from datetime import datetime, timezone
RUN = os.environ.get('PA3_RUN_TREE_V005', '0') == '1'
RESUME = os.environ.get('PA3_RESUME_TREE_V005', '0') == '1'
PROJECT = Path.cwd().resolve()
if not (PROJECT/'data').is_dir() and (PROJECT/'ventilator-pressure-prediction/data').is_dir():
    PROJECT = PROJECT/'ventilator-pressure-prediction'
NOTEBOOK = PROJECT/'tree_context_v005.ipynb'
DATA = PROJECT/'data'
PREV = PROJECT/'outputs/controlled_v003'
OUT = PROJECT/'outputs/improvements_v005/tree_context'
SEED = 20261102
THREADS = 4
INPUT_COLUMNS = ['id', 'breath_id', 'R', 'C', 'time_step', 'u_in', 'u_out']
DTYPES = dict(id='int64', breath_id='int32', R='int8', C='int8',
              time_step='float32', u_in='float32', u_out='int8', pressure='float32')
TREE_PARAMS = dict(n_estimators=2500, num_leaves=511, learning_rate=.04,
                   min_child_samples=20, max_bin=255, reg_lambda=2.)
FIT_PARAMS = dict(TREE_PARAMS, objective='regression_l1', subsample=.9,
                  subsample_freq=1, colsample_bytree=.9, n_jobs=THREADS,
                  random_state=SEED, deterministic=True, force_col_wise=True,
                  verbosity=-1, device_type='cpu')
ARMS = [('tree_offline', 'offline'), ('tree_dynamics', 'dynamics'),
        ('tree_combined', 'combined')]
PLAN = dict(version='tree_context_v005', seed=SEED, cpu_threads=THREADS,
            feature_arms=ARMS, fit_parameters=FIT_PARAMS,
            baseline='verified v003 full_screen_leaves_511; no new fit or inference',
            training_breaths=61114, validation_breaths=6791,
            training_rows='inspiration only (u_out == 0)',
            score='native-unit MAE on original A inspiration rows, exact saved ID order',
            validation_feedback=False, early_stopping=False, fixed_trees=2500,
            B_or_C_confirmation=False, final_all_fit=False, submission=False,
            offline_known_input_access=True, pressure_is_never_an_input=True)
print('Mode:', 'run fixed A tree comparisons' if RUN else 'review saved A tree comparisons')
print('Artifacts:', OUT)
if RUN:
    import importlib.metadata as metadata
    import lightgbm as lgb


In [ ]:
FEATURE_DEFINITION_SHA256 = '5b1d7e4dc891a517b880fde42b3e8ac6eaff6ebbbc6e50296f80b13cc895de78'


## Embedded feature definitions

These three definitions are copied exactly from cells tagged `v005-feature-definition` in `research/feature_components_v005.ipynb`, with only an execution guard added. The component notebook's 76 synthetic checks independently passed. The embedded source hash is verified when running.


In [ ]:
if RUN:
    import hashlib
    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd

    STEPS = 80
    V005_FEATURE_MODES = ('causal20', 'offline', 'dynamics', 'combined')
    V005_BASE_FEATURE_NAMES = (
        'R', 'C', 'time_step', 'u_in', 'u_out', 'step', 'dt',
        'control_sum', 'control_mean', 'control_integral',
        'u_in_lag_1', 'u_in_change_1', 'u_in_lag_2', 'u_in_change_2',
        'u_in_lag_4', 'u_in_change_4', 'u_in_lag_8', 'u_in_change_8',
        'u_in_mean_4', 'u_in_mean_8',
    )
    V005_OFFLINE_FEATURE_NAMES = (
        'u_in_lead_1', 'u_in_lead_2', 'u_in_lead_4', 'u_in_lead_8',
        'u_in_centered_mean_4', 'u_in_centered_mean_8',
        'inspiration_duration', 'inspiration_time_fraction',
        'time_to_u_out_switch', 'inspiration_u_in_peak',
        'inspiration_u_in_peak_time', 'inspiration_u_in_mean',
        'inspiration_u_in_std', 'inspiration_u_in_integral',
    )
    V005_EWM_TIMESCALES_SECONDS = (.05, .1, .2, .5, 1., 2.)
    V005_DYNAMICS_FEATURE_NAMES = (
        'rc_tau_seconds', 'time_over_rc_tau', 'u_in_times_R', 'u_in_over_R',
        'du_in_dt', 'control_trapezoid_integral',
        'control_trapezoid_over_compliance_L',
        'control_ewm_tau_0_05', 'control_ewm_tau_0_1',
        'control_ewm_tau_0_2', 'control_ewm_tau_0_5',
        'control_ewm_tau_1', 'control_ewm_tau_2',
    )


In [ ]:
if RUN:
    def features(frame: pd.DataFrame) -> tuple[np.ndarray, list[str], np.ndarray]:
        """Return features in original row order, after verifying 80-row blocks."""
        if len(frame) % STEPS:
            raise ValueError("Unexpected breath length; do not reshape this file")
        blocks = frame["breath_id"].to_numpy().reshape(-1, STEPS)
        breath_ids = blocks[:, 0]
        if not np.all(blocks == breath_ids[:, None]):
            raise ValueError("Breaths are not contiguous 80-row blocks")
        if len(np.unique(breath_ids)) != len(breath_ids):
            raise ValueError("Repeated breath blocks")
        if not frame["id"].is_unique:
            raise ValueError("Duplicate row IDs")
        u = frame["u_in"].to_numpy().reshape(-1, STEPS)
        t = frame["time_step"].to_numpy().reshape(-1, STEPS)
        if not np.all(np.diff(t, axis=1) > 0):
            raise ValueError("Timestamps must increase inside each breath")
        for column in ("R", "C"):
            values = frame[column].to_numpy().reshape(-1, STEPS)
            if not np.all(values == values[:, :1]):
                raise ValueError(f"{column} changes within a breath")

        columns = {}
        for column in ("R", "C", "time_step", "u_in", "u_out"):
            columns[column] = frame[column].to_numpy(dtype=np.float32)
        step = np.arange(STEPS)
        dt = np.diff(t, axis=1, prepend=t[:, :1])
        cumulative = np.cumsum(u, axis=1, dtype=np.float32)
        columns["step"] = np.broadcast_to(step, u.shape).ravel()
        columns["dt"] = dt.ravel()
        columns["control_sum"] = cumulative.ravel()
        columns["control_mean"] = (cumulative / (step + 1)).ravel()
        # Valve opening is not measured flow: this is a history proxy, not volume.
        columns["control_integral"] = np.cumsum(u * dt, axis=1).ravel()
        for lag in (1, 2, 4, 8):
            shifted = np.empty_like(u)
            shifted[:, :lag] = u[:, :1]
            shifted[:, lag:] = u[:, :-lag]
            columns[f"u_in_lag_{lag}"] = shifted.ravel()
            columns[f"u_in_change_{lag}"] = (u - shifted).ravel()
        padded_sum = np.concatenate([np.zeros((len(u), 1), dtype=np.float32), cumulative], axis=1)
        for window in (4, 8):
            begin = np.maximum(0, step + 1 - window)
            columns[f"u_in_mean_{window}"] = (
                (padded_sum[:, step + 1] - padded_sum[:, begin]) / (step + 1 - begin)
            ).ravel()
        names = list(columns)
        matrix = np.column_stack(list(columns.values())).astype(np.float32, copy=False)
        if not np.isfinite(matrix).all() or "pressure" in names:
            raise ValueError("Invalid features")
        return matrix, names, breath_ids


In [ ]:
if RUN:
    # Keep the legacy flat API available when importing these cells into a notebook.
    features_causal20 = features


    def _validate_feature_frame_v005(frame):
        """Validate known inputs only. Pressure and unrelated columns are never read."""
        if not isinstance(frame, pd.DataFrame):
            raise TypeError('Expected a pandas DataFrame')
        required = ('id', 'breath_id', 'R', 'C', 'time_step', 'u_in', 'u_out')
        if not frame.columns.is_unique:
            raise ValueError('Duplicate column names')
        missing = set(required) - set(frame.columns)
        if missing:
            raise ValueError(f'Missing required inputs: {sorted(missing)}')
        if not len(frame) or len(frame) % STEPS:
            raise ValueError('Expected nonempty contiguous 80-row breaths')
        for column in required:
            if not pd.api.types.is_numeric_dtype(frame[column]):
                raise ValueError(f'{column} must be numeric')
            values = frame[column].to_numpy()
            if not np.isfinite(values).all():
                raise ValueError(f'{column} must be finite')
        for column in ('id', 'breath_id'):
            values = frame[column].to_numpy()
            if not np.all(values == np.floor(values)):
                raise ValueError(f'{column} must contain integer IDs')
        if not frame['id'].is_unique:
            raise ValueError('Duplicate row IDs')
        blocks = frame['breath_id'].to_numpy().reshape(-1, STEPS)
        breath_ids = blocks[:, 0]
        if not np.all(blocks == breath_ids[:, None]):
            raise ValueError('Breaths are not contiguous 80-row blocks')
        if len(np.unique(breath_ids)) != len(breath_ids):
            raise ValueError('Repeated breath blocks')
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        if not np.all(np.diff(t, axis=1) > 0):
            raise ValueError('Timestamps must increase inside each breath')
        for column, levels in (('R', (5, 20, 50)), ('C', (10, 20, 50))):
            values = frame[column].to_numpy().reshape(-1, STEPS)
            if not np.all(values == values[:, :1]):
                raise ValueError(f'{column} changes within a breath')
            if not np.isin(values, levels).all():
                raise ValueError(f'Unexpected {column} level')
        out = frame['u_out'].to_numpy().reshape(-1, STEPS)
        if not np.isin(out, (0, 1)).all():
            raise ValueError('u_out must be binary')
        # The approved data has a single inspiration followed by expiration. A fully
        # observed inspiration without an expiration sample is supported as well.
        if np.any(out[:, 0] != 0) or np.any(np.diff(out.astype(np.int8), axis=1) < 0):
            raise ValueError('Expected initial inspiration followed by at most one valve opening')
        return breath_ids


    def feature_names_v005(feature_mode='causal20'):
        if feature_mode not in V005_FEATURE_MODES:
            raise ValueError(f'Unknown feature_mode: {feature_mode!r}')
        names = list(V005_BASE_FEATURE_NAMES)
        if feature_mode in ('offline', 'combined'):
            names.extend(V005_OFFLINE_FEATURE_NAMES)
        if feature_mode in ('dynamics', 'combined'):
            names.extend(V005_DYNAMICS_FEATURE_NAMES)
        return names


    def _offline_feature_block_v005(frame):
        """Known whole-breath input context; no target values or statistics."""
        u = frame['u_in'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        out = frame['u_out'].to_numpy().reshape(-1, STEPS)
        inspiration = out == 0
        n_breaths = len(u)
        columns = []
        for lead in (1, 2, 4, 8):
            shifted = np.empty_like(u)
            shifted[:, :-lead] = u[:, lead:]
            shifted[:, -lead:] = u[:, -1:]
            columns.append(shifted)
        prefix = np.concatenate((np.zeros((n_breaths, 1)), np.cumsum(u, axis=1)), axis=1)
        steps = np.arange(STEPS)
        for window in (4, 8):
            # Even centered windows have window//2 prior and window//2-1 future
            # neighbours, matching pandas rolling(center=True, min_periods=1).
            begin = np.maximum(0, steps - window//2)
            end = np.minimum(STEPS, steps + (window+1)//2)
            columns.append((prefix[:, end] - prefix[:, begin]) / (end-begin))
        has_expiration = np.any(out == 1, axis=1)
        first_expiration = np.argmax(out == 1, axis=1)
        switch_index = np.where(has_expiration, first_expiration, STEPS-1)
        switch_time = t[np.arange(n_breaths), switch_index]
        duration = switch_time - t[:, 0]
        # With strictly increasing 80-step timestamps and initial inspiration,
        # duration is positive, including the no-expiration boundary case.
        fraction = np.clip((t-t[:, :1])/duration[:, None], 0., 1.)
        until_switch = np.maximum(switch_time[:, None]-t, 0.)
        masked = np.where(inspiration, u, -np.inf)
        peak_index = np.argmax(masked, axis=1)  # first peak resolves a known-input tie
        peak = u[np.arange(n_breaths), peak_index]
        peak_time = t[np.arange(n_breaths), peak_index] - t[:, 0]
        count = inspiration.sum(axis=1)
        mean = np.sum(np.where(inspiration, u, 0.), axis=1)/count
        variance = np.sum(np.where(inspiration, (u-mean[:, None])**2, 0.), axis=1)/count
        std = np.sqrt(np.maximum(variance, 0.))
        # Integrate between inspiratory observations only; do not infer unobserved
        # flow or include the interval crossing into expiration.
        both_inspiration = inspiration[:, 1:] & inspiration[:, :-1]
        integral = np.sum(np.where(both_inspiration,
            .5*(u[:, 1:]+u[:, :-1])*np.diff(t, axis=1), 0.), axis=1)
        def broadcast(values):
            return np.broadcast_to(values[:, None], u.shape)
        columns.extend((broadcast(duration), fraction, until_switch, broadcast(peak),
            broadcast(peak_time), broadcast(mean), broadcast(std), broadcast(integral)))
        return np.stack(columns, axis=-1).astype(np.float32)


    def _dynamics_feature_block_v005(frame):
        """Causal timing and control proxies; valve opening is not measured flow."""
        u = frame['u_in'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        r = frame['R'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        c_litres = frame['C'].to_numpy(dtype=np.float64).reshape(-1, STEPS)/1000.
        dt = np.diff(t, axis=1, prepend=t[:, :1])
        tau = r*c_litres  # R in cmH2O/(L/s), C converted from mL/cmH2O
        derivative = np.zeros_like(u)
        derivative[:, 1:] = np.diff(u, axis=1)/dt[:, 1:]
        trapezoid = np.zeros_like(u)
        trapezoid[:, 1:] = np.cumsum(.5*(u[:, 1:]+u[:, :-1])*dt[:, 1:], axis=1)
        # These interactions can help the learner represent R/C sensitivity. They
        # are not asserted to have physical pressure/flow/volume units because u_in
        # is a valve-opening percentage, not observed flow or driving pressure.
        columns = [tau, (t-t[:, :1])/tau, u*r, u/r, derivative,
                   trapezoid, trapezoid/c_litres]
        scales = np.asarray(V005_EWM_TIMESCALES_SECONDS, dtype=np.float64)
        histories = np.empty((*u.shape, len(scales)), dtype=np.float32)
        state = np.broadcast_to(u[:, :1], (len(u), len(scales))).copy()
        histories[:, 0, :] = state
        # One loop over 80 timesteps; all breaths and timescales run together.
        # Irregular-time EWMA includes the control observed at the current step.
        for step in range(1, STEPS):
            alpha = np.exp(-dt[:, step, None]/scales[None, :])
            state = alpha*state + (1.-alpha)*u[:, step, None]
            histories[:, step, :] = state
        fixed = np.stack(columns, axis=-1).astype(np.float32)
        return np.concatenate((fixed, histories), axis=-1)


    def make_sequence_features(frame, feature_mode='causal20'):
        """Return (X[breaths,80,F] float32, names, breath_ids), in input order.

        causal20: exact legacy features; offline: legacy + known future/context;
        dynamics: legacy + causal dynamics proxies; combined: all three blocks.
        No scaling, fitting, target reads, IDs as features, or frame mutation occur.
        """
        names = feature_names_v005(feature_mode)
        breath_ids = _validate_feature_frame_v005(frame)
        base_flat, base_names, old_ids = features_causal20(frame)
        if base_names != list(V005_BASE_FEATURE_NAMES) or not np.array_equal(breath_ids, old_ids):
            raise ValueError('Legacy feature schema changed')
        base = base_flat.reshape(-1, STEPS, len(base_names))
        if feature_mode == 'causal20':
            return base, names, breath_ids.copy()
        result = np.empty((len(breath_ids), STEPS, len(names)), dtype=np.float32)
        result[..., :len(base_names)] = base
        position = len(base_names)
        if feature_mode in ('offline', 'combined'):
            block = _offline_feature_block_v005(frame)
            result[..., position:position+block.shape[-1]] = block
            position += block.shape[-1]
        if feature_mode in ('dynamics', 'combined'):
            block = _dynamics_feature_block_v005(frame)
            result[..., position:position+block.shape[-1]] = block
            position += block.shape[-1]
        if position != len(names) or not np.isfinite(result).all():
            raise ValueError('Invalid feature output')
        return result, names, breath_ids.copy()


In [ ]:
def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def array_hash(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name+'.tmp')
    with temporary.open('w') as stream:
        json.dump(value, stream, indent=2, allow_nan=False)
        stream.write('\n')
    os.replace(temporary, path)

def save_array(path, value):
    path = Path(path)
    temporary = path.with_name(path.name+'.tmp')
    with temporary.open('wb') as stream:
        np.save(stream, value, allow_pickle=False)
    os.replace(temporary, path)

def archive_paths(paths, label):
    paths = [Path(p) for p in paths if Path(p).exists()]
    if not paths:
        return None
    name = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'_'+uuid.uuid4().hex[:8]
    destination = OUT/'archive'/(label+'_'+name)
    destination.mkdir(parents=True, exist_ok=False)
    for path in paths:
        shutil.move(str(path), str(destination/path.name))
    print('Preserved incomplete artifacts:', destination, flush=True)
    return destination

def event(kind, **values):
    with (OUT/'events.jsonl').open('a') as stream:
        stream.write(json.dumps(dict(timestamp_utc=datetime.now(timezone.utc).isoformat(),
                                     event=kind, **values), allow_nan=False)+'\n')

def group_indices(breath_ids, saved_ids):
    index = pd.Index(breath_ids)
    saved_ids = np.asarray(saved_ids)
    if len(np.unique(saved_ids)) != len(saved_ids):
        raise ValueError('Duplicate saved split breath IDs')
    result = np.sort(index.get_indexer(saved_ids))
    if (result < 0).any():
        raise ValueError('Saved split contains unknown breath IDs')
    return result

def scored_features(frame, indices, feature_mode, u_out, expected_breath_ids):
    # Only whole breaths are selected, preserving 80-step feature boundaries.
    row_positions = (indices[:, None]*80+np.arange(80)[None, :]).ravel()
    subset = frame.iloc[row_positions]
    sequence, names, ids = make_sequence_features(subset, feature_mode)
    if not np.array_equal(ids, expected_breath_ids[indices]):
        raise ValueError('Feature breath order mismatch')
    result = np.ascontiguousarray(sequence[u_out[indices] == 0], dtype=np.float32)
    del subset, sequence, row_positions
    gc.collect()
    return result, names

def score_arrays(prediction, target, row_ids, expected_ids):
    prediction = np.asarray(prediction, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    if not np.array_equal(row_ids, expected_ids):
        raise ValueError('Scored row IDs are misaligned')
    if prediction.shape != target.shape or prediction.ndim != 1:
        raise ValueError('Prediction shape mismatch')
    if not np.isfinite(prediction).all() or not np.isfinite(target).all():
        raise ValueError('Nonfinite predictions or labels')
    return float(np.abs(prediction-target).mean()), np.abs(prediction-target)

def trial_paths(tag):
    return dict(record=OUT/('record_'+tag+'.json'),
                checkpoint=OUT/(tag+'.txt'),
                prediction=OUT/(tag+'_predictions.npy'),
                by_rc=OUT/(tag+'_by_rc.csv'),
                progress=OUT/(tag+'_progress.jsonl'))

def refresh_table():
    # One notebook worker, one arm at a time; no concurrent CSV writers.
    rows = []
    for path in sorted(OUT.glob('record_A_tree_*.json')):
        record = json.loads(path.read_text())
        if record.get('status') == 'complete':
            rows.append({key:record.get(key) for key in
                ['name','split','kind','feature_mode','n_features','seed','training_breaths',
                 'validation_breaths','scored_points','validation_mae','fit_seconds',
                 'runtime_seconds','reused_checkpoint','checkpoint_sha256','prediction_sha256']})
    temporary = OUT/'results.csv.tmp'
    pd.DataFrame(rows).to_csv(temporary, index=False)
    os.replace(temporary, OUT/'results.csv')

def write_breakdown(path, absolute_error):
    table = pd.DataFrame(dict(R=A_RC[:, 0], C=A_RC[:, 1], absolute_error=absolute_error))
    summary = table.groupby(['R','C']).absolute_error.agg(['mean','count','sum']).reset_index()
    summary = summary.rename(columns=dict(mean='mae', count='scored_points', sum='total_absolute_error'))
    temporary = path.with_name(path.name+'.tmp')
    summary.to_csv(temporary, index=False)
    os.replace(temporary, path)
    return summary.to_dict(orient='records')

def validate_complete(record, identity):
    if record.get('status') != 'complete' or record.get('identity') != identity:
        raise ValueError('Completed trial identity mismatch; preserve it and use a new output version')
    if record.get('identity_sha256') != canonical_hash(identity):
        raise ValueError('Completed identity hash mismatch')
    for path_key, hash_key in [('checkpoint_path','checkpoint_sha256'),
                               ('prediction_file','prediction_sha256'),
                               ('scored_row_id_file','scored_row_id_sha256'),
                               ('rc_breakdown_file','rc_breakdown_sha256')]:
        if digest(Path(record[path_key])) != record[hash_key]:
            raise ValueError('Completed artifact hash mismatch: '+path_key)
    prediction = np.load(record['prediction_file'], allow_pickle=False)
    row_ids = np.load(record['scored_row_id_file'], allow_pickle=False)
    mae, error = score_arrays(prediction, A_TARGET, row_ids, A_ROW_IDS)
    if abs(mae-record['validation_mae']) > 1e-12:
        raise ValueError('Completed MAE mismatch')
    return record

def trial_identity(name, mode, reused=False):
    return dict(name=name, split='A', feature_mode=mode,
                input_columns=feature_names_v005(mode),
                seed=SEED, fit_parameters=FIT_PARAMS, fixed_trees=2500,
                design_sha256=canonical_hash(DESIGN),
                fit_breath_sha256=DESIGN['group_hashes']['A_fit'],
                validation_breath_sha256=DESIGN['group_hashes']['A_validation'],
                scored_row_id_sha256=DESIGN['scored_row_id_sha256'],
                reused_checkpoint=reused)

def maybe_reuse(tag, identity):
    paths = trial_paths(tag)
    if paths['record'].is_file():
        record = json.loads(paths['record'].read_text())
        if record.get('status') == 'complete':
            if not RESUME:
                raise ValueError('Completed trial exists; set PA3_RESUME_TREE_V005=1')
            record = validate_complete(record, identity)
            print('REUSE', tag, record['validation_mae'], flush=True)
            return record
    # Partial files are never silently overwritten; archive all names and tmp files.
    incomplete = list(paths.values())+[p.with_name(p.name+'.tmp') for p in paths.values()]
    incomplete += [paths['checkpoint'].with_name(paths['checkpoint'].name+'.tmp.txt')]
    archive_paths(incomplete, tag)
    return None


## Lock approved data, original A groups, source and environment

The training CSV hash must match the existing approved audit. No new split is drawn. The full notebook code, embedded feature code, prior design, prior split, baseline weights/predictions/row IDs, training/validation groups, parameters, software versions and exact feature names enter the design lock. Compatible resume validates every completed checkpoint, prediction, row-ID file and recomputed MAE. Incomplete per-arm artifacts move to a timestamped archive before retrying.


In [ ]:
if RUN:
    OUT.mkdir(parents=True, exist_ok=True)
    RUN_LOCK = (OUT/'.run.lock').open('a')
    try:
        fcntl.flock(RUN_LOCK.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        raise RuntimeError('Another tree_context_v005 worker owns the run lock')
    quality = json.loads((PROJECT/'quality_audit_v002.json').read_text())
    train_sha = digest(DATA/'train.csv')
    assert train_sha == quality['files']['train.csv']['sha256']
    assert train_sha == '2dc30dc5c8e603c8c68200830eab7ed8c7af42c823f8ae68b0beb663eea817d7'
    source_notebook = json.loads(NOTEBOOK.read_text())
    code_source = '\n'.join(''.join(c['source']) for c in source_notebook['cells'] if c['cell_type']=='code')
    source_sha = hashlib.sha256(code_source.encode()).hexdigest()
    import textwrap
    embedded = [textwrap.dedent(''.join(c['source']).split('\n', 1)[1]).rstrip()+'\n'
                for c in source_notebook['cells']
                if 'v005-feature-definition' in c.get('metadata',{}).get('tags',[])]
    assert len(embedded) == 3
    assert hashlib.sha256('\n'.join(embedded).encode()).hexdigest() == FEATURE_DEFINITION_SHA256
    frame = pd.read_csv(DATA/'train.csv', dtype=DTYPES)
    BREATH_IDS = _validate_feature_frame_v005(frame)
    assert len(BREATH_IDS) == 67905
    Y = frame.pressure.to_numpy(copy=True).reshape(-1,80)
    UOUT = frame.u_out.to_numpy(copy=True).reshape(-1,80)
    ROW_IDS = frame.id.to_numpy(copy=True).reshape(-1,80)
    RC = frame[['R','C']].to_numpy(copy=True)[::80]
    KNOWN_FRAME = frame[INPUT_COLUMNS].copy()
    del frame
    with np.load(PREV/'split_breath_ids.npz', allow_pickle=False) as saved:
        GROUPS = {key:group_indices(BREATH_IDS, saved[key]) for key in ['A_fit','A_validation']}
    assert len(GROUPS['A_fit']) == 61114 and len(GROUPS['A_validation']) == 6791
    assert not np.intersect1d(GROUPS['A_fit'], GROUPS['A_validation']).size
    assert np.array_equal(np.sort(np.concatenate(list(GROUPS.values()))), np.arange(67905))
    A_MASK = UOUT[GROUPS['A_validation']] == 0
    A_TARGET = Y[GROUPS['A_validation']][A_MASK].astype(np.float64)
    A_ROW_IDS = ROW_IDS[GROUPS['A_validation']][A_MASK]
    A_RC = np.repeat(RC[GROUPS['A_validation']], 80, axis=0)[A_MASK.ravel()]
    assert len(A_TARGET) == 206098
    OLD_RECORD = json.loads((PREV/'full_screen_leaves_511.json').read_text())
    OLD_DESIGN = json.loads((PREV/'design.json').read_text())
    OLD_CHECKPOINT = PREV/'full_screen_leaves_511.txt'
    OLD_PREDICTION = PREV/'full_screen_leaves_511_validation_predictions.npy'
    OLD_ROW_IDS = PREV/'A_validation_row_ids.npy'
    assert OLD_RECORD['status'] == 'complete' and OLD_RECORD['split'] == 'A'
    assert OLD_RECORD['definition'] == dict(family='LightGBM', params=TREE_PARAMS, history='both', rc='numeric')
    assert OLD_RECORD['seed'] == SEED and OLD_RECORD['training_breaths'] == 61114
    assert OLD_RECORD['validation_breaths'] == 6791 and OLD_RECORD['best_iteration'] == 2500
    assert OLD_RECORD['input_columns'] == feature_names_v005('causal20')
    assert OLD_RECORD['identity_sha256'] == canonical_hash(OLD_RECORD['identity'])
    assert OLD_RECORD['identity']['design_sha256'] == canonical_hash(OLD_DESIGN)
    assert OLD_DESIGN['train_sha256'] == train_sha
    group_hashes = {key:array_hash(BREATH_IDS[value]) for key,value in GROUPS.items()}
    assert group_hashes['A_fit'] == OLD_RECORD['identity']['fit_breath_sha256'] == OLD_DESIGN['groups']['Afit']
    assert group_hashes['A_validation'] == OLD_RECORD['identity']['val_breath_sha256'] == OLD_DESIGN['groups']['Aval']
    assert digest(OLD_CHECKPOINT) == OLD_RECORD['model_sha256'] == '648d426cf4fd50aba03ff396482ecd4238dea4a6f1d64d98a25d29e81dd17088'
    assert digest(OLD_PREDICTION) == '5c59ad8b78d20c66bb14ed87ec241a0df8ff56dc84d53eee5c2fd20ff0865578'
    assert digest(OLD_ROW_IDS) == '203c2ec1a2650798779225d4d693780f61ba26a19a64e660c0d99e916e790748'
    assert np.array_equal(np.load(OLD_ROW_IDS, allow_pickle=False), A_ROW_IDS)
    old_prediction = np.load(OLD_PREDICTION, allow_pickle=False)
    old_mae, _ = score_arrays(old_prediction, A_TARGET, A_ROW_IDS, A_ROW_IDS)
    assert abs(old_mae-OLD_RECORD['validation_mae']) < 1e-12
    baseline_model = lgb.Booster(model_file=str(OLD_CHECKPOINT))
    assert baseline_model.feature_name() == feature_names_v005('causal20')
    assert baseline_model.num_trees() == 2500
    del baseline_model, old_prediction
    DESIGN = json.loads(json.dumps(dict(plan=PLAN, source_sha256=source_sha,
        feature_definition_sha256=FEATURE_DEFINITION_SHA256,
        feature_names={mode:feature_names_v005(mode) for mode in V005_FEATURE_MODES},
        train_sha256=train_sha, previous_design_sha256=digest(PREV/'design.json'),
        previous_split_sha256=digest(PREV/'split_breath_ids.npz'),
        group_hashes=group_hashes, scored_row_id_sha256=digest(OLD_ROW_IDS),
        baseline_record_sha256=digest(PREV/'full_screen_leaves_511.json'),
        baseline_checkpoint_sha256=digest(OLD_CHECKPOINT),
        baseline_prediction_sha256=digest(OLD_PREDICTION),
        versions={name:metadata.version(name) for name in ['lightgbm','numpy','pandas']})))
    design_path = OUT/'design.json'
    if design_path.exists():
        assert RESUME, 'Existing design requires PA3_RESUME_TREE_V005=1'
        assert json.loads(design_path.read_text()) == DESIGN, 'Source, design, environment or data changed'
        assert digest(OUT/'A_validation_row_ids.npy') == DESIGN['scored_row_id_sha256']
    else:
        # If initialization previously failed before its lock was committed, retain remnants.
        archive_paths([p for p in OUT.iterdir() if p.name not in ['archive','.run.lock']], 'initialization')
        save_array(OUT/'A_validation_row_ids.npy', A_ROW_IDS)
        assert digest(OUT/'A_validation_row_ids.npy') == DESIGN['scored_row_id_sha256']
        temporary = OUT/'split_breath_ids.npz.tmp'
        with temporary.open('wb') as stream:
            np.savez_compressed(stream, **{key:BREATH_IDS[value] for key,value in GROUPS.items()})
        os.replace(temporary, OUT/'split_breath_ids.npz')
        write_json(design_path, DESIGN)
    event('initialized', resume=RESUME, design_sha256=canonical_hash(DESIGN),
          source_sha256=source_sha, groups={key:len(value) for key,value in GROUPS.items()})
    print('Locked A groups:', {key:len(value) for key,value in GROUPS.items()}, flush=True)
    gc.collect()


In [ ]:
def complete_record(name, mode, identity, paths, mae, breakdown, fit_seconds,
                    runtime_seconds, reused, resolved_parameters):
    record = dict(record_type='trial', status='complete', name=name, trial='A_'+name,
        split='A', kind='reference_tree' if reused else 'fresh_tree',
        feature_mode=mode, n_features=len(feature_names_v005(mode)),
        input_columns=feature_names_v005(mode), categorical_columns=[], seed=SEED,
        recipe=dict(name=name, kind='reference_tree' if reused else 'fresh_tree',
                    feature_mode=mode, seed=SEED, params=TREE_PARAMS,
                    fit_params=FIT_PARAMS, fixed_trees=2500, rc='numeric'),
        identity=identity, identity_sha256=canonical_hash(identity),
        design_sha256=canonical_hash(DESIGN), source_sha256=DESIGN['source_sha256'],
        training_breaths=61114, validation_breaths=6791, scored_points=len(A_TARGET),
        validation_mae=mae, fit_seconds=fit_seconds, runtime_seconds=runtime_seconds,
        elapsed_seconds=runtime_seconds, reused_checkpoint=reused,
        best_iteration=2500, best_epoch=None, validation_feedback=False,
        checkpoint_path=str(OLD_CHECKPOINT if reused else paths['checkpoint']),
        prediction_file=str(paths['prediction']),
        scored_row_id_file=str(OUT/'A_validation_row_ids.npy'),
        rc_breakdown_file=str(paths['by_rc']), rc_breakdown=breakdown,
        progress_file=None if reused else str(paths['progress']),
        termination='verified_prior_checkpoint' if reused else 'fixed_tree_budget',
        resolved_parameters=resolved_parameters)
    for path_key, hash_key in [('checkpoint_path','checkpoint_sha256'),
                               ('prediction_file','prediction_sha256'),
                               ('scored_row_id_file','scored_row_id_sha256'),
                               ('rc_breakdown_file','rc_breakdown_sha256')]:
        record[hash_key] = digest(record[path_key])
    # Old field aliases keep downstream result readers compatible.
    record['model_file'] = record['checkpoint_path']
    record['model_sha256'] = record['checkpoint_sha256']
    if reused:
        record['original_fit_seconds'] = OLD_RECORD['fit_seconds']
        record['original_runtime_seconds'] = OLD_RECORD['runtime_seconds']
        record['original_record_file'] = str(PREV/'full_screen_leaves_511.json')
    write_json(paths['record'], record)
    refresh_table()
    event('trial_completed', name=name, validation_mae=mae, fit_seconds=fit_seconds,
          runtime_seconds=runtime_seconds, reused_checkpoint=reused)
    return record

def reuse_baseline():
    name, mode = 'tree_causal20', 'causal20'
    tag = 'A_'+name
    identity = trial_identity(name, mode, reused=True)
    existing = maybe_reuse(tag, identity)
    if existing is not None:
        return existing
    started = time.monotonic()
    paths = trial_paths(tag)
    prediction = np.load(OLD_PREDICTION, allow_pickle=False)
    mae, error = score_arrays(prediction, A_TARGET, A_ROW_IDS, A_ROW_IDS)
    save_array(paths['prediction'], prediction)
    assert digest(paths['prediction']) == DESIGN['baseline_prediction_sha256']
    breakdown = write_breakdown(paths['by_rc'], error)
    result = complete_record(name, mode, identity, paths, mae, breakdown, 0.,
        time.monotonic()-started, True, OLD_RECORD['resolved_parameters'])
    print('VERIFIED BASELINE', name, mae, flush=True)
    return result

def fit_tree_A(name, mode):
    tag = 'A_'+name
    identity = trial_identity(name, mode)
    existing = maybe_reuse(tag, identity)
    if existing is not None:
        return existing
    paths = trial_paths(tag)
    started = time.monotonic()
    event('trial_started', name=name, feature_mode=mode, identity_sha256=canonical_hash(identity))
    print('START', name, mode, 'features', len(feature_names_v005(mode)), flush=True)
    learner = booster = trainx = valx = None
    try:
        trainx, names = scored_features(KNOWN_FRAME, GROUPS['A_fit'], mode, UOUT, BREATH_IDS)
        trainy = Y[GROUPS['A_fit']][UOUT[GROUPS['A_fit']] == 0]
        assert trainx.shape == (len(trainy), len(names)) and names == identity['input_columns']
        learner = lgb.LGBMRegressor(**FIT_PARAMS)
        fit_started = time.monotonic()
        def progress(environment):
            iteration = environment.iteration+1
            if iteration % 250 == 0:
                item = dict(timestamp_utc=datetime.now(timezone.utc).isoformat(),
                    name=name, iteration=iteration, fixed_trees=2500,
                    elapsed_fit_seconds=time.monotonic()-fit_started,
                    validation_feedback=False)
                with paths['progress'].open('a') as stream:
                    stream.write(json.dumps(item, allow_nan=False)+'\n')
                print('TREE PROGRESS', name, iteration, round(item['elapsed_fit_seconds'], 2), flush=True)
        progress.order = 10
        progress.before_iteration = False
        # No eval_set or validation labels enter fit; 2500 trees are unconditional.
        learner.fit(trainx, trainy, feature_name=names, callbacks=[progress])
        fit_seconds = time.monotonic()-fit_started
        assert learner.booster_.num_trees() == 2500
        temporary_model = paths['checkpoint'].with_name(paths['checkpoint'].name+'.tmp.txt')
        learner.booster_.save_model(str(temporary_model), num_iteration=2500)
        os.replace(temporary_model, paths['checkpoint'])
        del trainx, trainy, learner
        trainx = learner = None
        gc.collect()
        # Reload serialized weights. Construct and predict only scored A rows.
        booster = lgb.Booster(model_file=str(paths['checkpoint']))
        assert booster.feature_name() == names and booster.num_trees() == 2500
        valx, valnames = scored_features(KNOWN_FRAME, GROUPS['A_validation'], mode, UOUT, BREATH_IDS)
        assert valnames == names and len(valx) == len(A_ROW_IDS)
        prediction = booster.predict(valx, num_iteration=2500, num_threads=THREADS)
        mae, error = score_arrays(prediction, A_TARGET, A_ROW_IDS, A_ROW_IDS)
        save_array(paths['prediction'], np.asarray(prediction, dtype=np.float64))
        breakdown = write_breakdown(paths['by_rc'], error)
        record = complete_record(name, mode, identity, paths, mae, breakdown,
            fit_seconds, time.monotonic()-started, False, booster.params)
        print('COMPLETE', name, 'A MAE', mae, 'seconds', record['runtime_seconds'], flush=True)
        return record
    except Exception as error:
        event('trial_failed', name=name, error=repr(error), elapsed_seconds=time.monotonic()-started)
        raise
    finally:
        del trainx, valx, learner, booster
        gc.collect()


## Verified baseline and sequential new fits

One CPU worker uses at most four LightGBM threads. The callback logs elapsed fit time every 250 trees and does not inspect validation data. Each record is committed only after checkpoint serialization, scored-row prediction, finite/alignment/MAE checks, and R/C breakdown generation. No partial model is resumed: retries archive its files and restart the same fixed recipe.


In [ ]:
if RUN:
    A_RECORDS = {}
    A_RECORDS['tree_causal20'] = reuse_baseline()


In [ ]:
if RUN:
    A_RECORDS['tree_offline'] = fit_tree_A('tree_offline', 'offline')


In [ ]:
if RUN:
    A_RECORDS['tree_dynamics'] = fit_tree_A('tree_dynamics', 'dynamics')


In [ ]:
if RUN:
    A_RECORDS['tree_combined'] = fit_tree_A('tree_combined', 'combined')


In [ ]:
if RUN:
    refresh_table()
    event('A_tree_comparisons_complete', records={name:record['validation_mae'] for name,record in A_RECORDS.items()},
          new_model_fits=3, B_or_C_confirmation=False, final_all_fit=False)
    fcntl.flock(RUN_LOCK.fileno(), fcntl.LOCK_UN)
    RUN_LOCK.close()
    print('A-only fixed-budget comparisons complete. Validation selection remains exploratory.', flush=True)

records = []
if OUT.is_dir():
    for path in sorted(OUT.glob('record_A_tree_*.json')):
        record = json.loads(path.read_text())
        if record.get('status') == 'complete':
            records.append({key:record.get(key) for key in
                ['name','feature_mode','n_features','validation_mae','fit_seconds','runtime_seconds','reused_checkpoint']})
if records:
    for record in records:
        print(json.dumps(record, sort_keys=True))
else:
    print('Prepared three A feature arms; no saved v005 tree results. Set PA3_RUN_TREE_V005=1 to execute.')
